# 🕌 Islomiy Banklarda Moliyaviy Xizmatlar Riskini Baholash
## Statistik va Machine Learning Yondashuvi — **v3.0 (Standalone)**

> **Faqat standart kutubxonalar:** `numpy · pandas · matplotlib · seaborn · sklearn · scipy`

---

| # | Bo'lim | Tavsif |
|---|--------|--------|
| 1 | Dataset | 2500 tranzaksiya, 32 feature, AAOIFI/IFSB asosida |
| 2 | EDA | Vizualizatsiya, taqsimlash, korrelyatsiya |
| 3 | Statistik testlar | KS, Chi-square, VIF, Normallik, Mutual Info |
| 4 | Risk metrikalar | VaR, CVaR, Sharpe, Sortino, SRI, Basel EL/UL |
| 5 | Monte Carlo | 20 000 senariy, GBM, Stress Testing |
| 6 | ML + SMOTE | LogReg, RF, GBM, class_weight + manual SMOTE |
| 7 | GridSearchCV | RF + GBM hyperparameter tuning (Optuna o'rniga) |
| 8 | Interpretability | Permutation Importance + PDP (SHAP o'rniga) |
| 9 | Threshold Opt | Precision-Recall, Optimal Cutoff |
| 10 | Ensemble | Soft Voting + Bootstrap 95% CI |
| 11 | Portfel | Efficient Frontier, Max Sharpe |
| 12 | Xulosalar | Tavsiyalar, hisobot |

In [8]:
# ═══════════════════════════════════════════════════════════════
#  CELL 0 — Kutubxonalar va Global Sozlamalar
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score,
    precision_score, recall_score, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.pipeline import Pipeline

# Scipy
from scipy import stats
from scipy.stats import (
    kstest, chi2_contingency, shapiro,
    jarque_bera, pearsonr, spearmanr
)
from scipy.optimize import minimize

# Global uslub
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#1B4F72','#A93226','#1E8449','#D4A017','#6C3483']
plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.titleweight': 'bold'
})
np.random.seed(42)

print('✅ Barcha kutubxonalar yuklandi (faqat standart paket)')
print('   numpy | pandas | matplotlib | seaborn | sklearn | scipy')

ModuleNotFoundError: No module named 'matplotlib'

---
## 📊 1-Bo'lim: Dataset Generatsiyasi (AAOIFI / IFSB)

In [9]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Sintetik Dataset (2500 tranzaksiya, 32 feature)
# ═══════════════════════════════════════════════════════════════
N = 2500

SVC_DIST  = {'Murabaha':0.42,'Musharaka':0.22,'Ijara':0.21,'Sukuk':0.15}
SVC_LIST  = list(SVC_DIST.keys())
RISK_P = {
    'Murabaha':  {'pd':0.082,'lgd':0.45,'vol':0.118,'rate':(0.07,0.18),'tenor':[6,12,24,36,60]},
    'Musharaka': {'pd':0.148,'lgd':0.60,'vol':0.245,'rate':(0.12,0.38),'tenor':[12,24,36,60,84,120]},
    'Ijara':     {'pd':0.062,'lgd':0.35,'vol':0.098,'rate':(0.05,0.13),'tenor':[12,24,36,60,84]},
    'Sukuk':     {'pd':0.038,'lgd':0.28,'vol':0.078,'rate':(0.04,0.10),'tenor':[24,36,60,84,120]},
}
REGIONS  = ['Toshkent','Samarqand',"Farg'ona",'Buxoro','Namangan','Qashqadaryo']
SECTORS  = ['Savdo','Ishlab chiqarish',"Qishloq xo'jaligi",'Qurilish','Xizmat','Eksport']

svc_arr = np.random.choice(SVC_LIST, N, p=list(SVC_DIST.values()))
rows = []

for svc in svc_arr:
    p = RISK_P[svc]

    # ─ Mijoz
    cscore   = np.clip(np.random.normal(645,85), 300, 850)
    age      = np.random.randint(22, 65)
    exp_yr   = np.random.randint(1, 25)
    loan_amt = np.random.lognormal(10.8, 1.1)
    tenor    = np.random.choice(p['tenor'])
    rate     = np.random.uniform(*p['rate'])
    region   = np.random.choice(REGIONS)
    sector   = np.random.choice(SECTORS)

    # ─ Risk omillari
    ltv      = np.clip(np.random.beta(4.5,3), 0.10, 0.95)
    dsr      = np.clip(np.random.beta(3,5),   0.05, 0.85)
    col_q    = np.random.choice([1,2,3,4,5], p=[0.05,0.20,0.40,0.25,0.10])
    liq      = np.clip(np.random.beta(7,2.5), 0.20, 1.0)
    lev      = np.random.uniform(0.10, 0.90)
    n_prev   = np.random.poisson(2.5)
    n_def    = np.random.binomial(n_prev, 0.08) if n_prev > 0 else 0

    # ─ Sharia omillari
    sharia   = np.clip(np.random.beta(8,2.5),  0.55, 1.0)
    gharar   = np.clip(np.random.beta(2,8),    0.00, 0.50)
    maysir   = np.clip(np.random.beta(1.5,9),  0.00, 0.40)
    halal    = np.random.choice([0,1], p=[0.12,0.88])

    # ─ Makro
    mkt_vol  = np.abs(np.random.normal(p['vol'], 0.025))
    gdp_g    = np.random.normal(0.056, 0.014)
    inf_r    = np.random.normal(0.098, 0.022)
    fx       = np.abs(np.random.normal(0.048, 0.028))
    oil      = np.random.normal(0.02, 0.15)
    bidx     = np.random.normal(0.04, 0.08)

    # ─ Default ehtimoli (logit)
    z = (-4.2
         + p['pd']*12
         - (cscore-650)*0.006
         + ltv*2.5 + dsr*3.8
         - sharia*2.0 + gharar*4.0 + maysir*3.5
         + inf_r*6.0 + mkt_vol*5.5
         - liq*2.5 + lev*1.8
         + n_def*0.9 - halal*0.6
         - (col_q-3)*0.5
         + np.random.normal(0, 0.25))
    pd_val  = 1/(1+np.exp(-z))
    is_def  = int(np.random.random() < pd_val)

    ead = loan_amt*(1+rate*tenor/12*0.5)
    lgd = np.clip(np.random.normal(p['lgd'],0.08), 0.10, 0.95)
    el  = pd_val*ead*lgd

    if pd_val < 0.10:  rlvl,rcode = 'Past',0
    elif pd_val < 0.25: rlvl,rcode = "O'rta",1
    elif pd_val < 0.45: rlvl,rcode = 'Yuqori',2
    else:               rlvl,rcode = 'Juda Yuqori',3

    rows.append({
        'xizmat_turi':svc,'mintaqa':region,'sektor':sector,
        'kredit_ball':round(cscore),'yosh':age,'tajriba':exp_yr,
        'oldingi_kreditlar':n_prev,'oldingi_defaultlar':n_def,
        'moliyalash_miqdori':round(loan_amt),'muddat_oy':tenor,
        'foyda_stavkasi':round(rate,4),'ltv_nisbati':round(ltv,4),
        'qarz_xizmat_nisbati':round(dsr,4),'likvidlik':round(liq,4),
        'leverage':round(lev,4),'garov_sifati':col_q,
        'sharia_audit':round(sharia,4),'gharar_darajasi':round(gharar,4),
        'maysir_ekspozitsiya':round(maysir,4),'halal_sertifikat':halal,
        'bozor_volatilligi':round(mkt_vol,4),'yim_osishi':round(gdp_g,4),
        'inflyatsiya':round(inf_r,4),'valyuta_tebranishi':round(fx,4),
        'neft_narxi':round(oil,4),'bank_indeksi':round(bidx,4),
        'pd_qiymati':round(pd_val,4),'ead':round(ead),'lgd':round(lgd,4),
        'kutilgan_zarar':round(el),'default_holati':is_def,
        'risk_darajasi':rlvl,'risk_kodi':rcode
    })

df = pd.DataFrame(rows)

# Label encoding
for col,name in [('xizmat_turi','xizmat_enc'),('mintaqa','mintaqa_enc'),('sektor','sektor_enc')]:
    le = LabelEncoder()
    df[name] = le.fit_transform(df[col])

print(f'✅ Dataset: {df.shape[0]} qator × {df.shape[1]} ustun')
print(f'Default nisbati: {df["default_holati"].mean():.2%}')
print()
summary = df.groupby('xizmat_turi')['default_holati'].agg(['count','mean','sum'])
summary.columns = ['N','Default%','Default_N']
summary['Default%'] = summary['Default%'].round(4)
print(summary)
df.head(3)

NameError: name 'LabelEncoder' is not defined

---
## 🔍 2-Bo'lim: Kengaytirilgan EDA

In [11]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2 — EDA: 8 grafik bir sahifada
# ═══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20,15))
gs  = gridspec.GridSpec(3,4,hspace=0.50,wspace=0.38)

# 1. Pie
ax = fig.add_subplot(gs[0,0])
cnt = df['xizmat_turi'].value_counts()
ax.pie(cnt,labels=cnt.index,autopct='%1.1f%%',colors=PALETTE,
       startangle=90,wedgeprops={'linewidth':2,'edgecolor':'white'})
ax.set_title('Xizmat Turlari Ulushi')

# 2. Default nisbati
ax = fig.add_subplot(gs[0,1])
dr = df.groupby('xizmat_turi')['default_holati'].mean().sort_values(ascending=False)
bars = ax.bar(dr.index,dr.values*100,color=PALETTE[:4],edgecolor='white',lw=1.5)
for b,v in zip(bars,dr.values):
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,f'{v:.1%}',
            ha='center',fontsize=9,fontweight='bold')
ax.set_title('Default Nisbati'); ax.set_ylabel('%')

# 3. Violin — PD
ax = fig.add_subplot(gs[0,2])
data_v = [df[df['xizmat_turi']==s]['pd_qiymati'].values for s in SVC_LIST]
parts = ax.violinplot(data_v,showmedians=True,showextrema=True)
for i,pc in enumerate(parts['bodies']):
    pc.set_facecolor(PALETTE[i]); pc.set_alpha(0.75)
ax.set_xticks([1,2,3,4]); ax.set_xticklabels(SVC_LIST,fontsize=8)
ax.set_title('PD Violin'); ax.set_ylabel('Default Ehtimoli')

# 4. Box — Kredit ball
ax = fig.add_subplot(gs[0,3])
grp = [df[df['xizmat_turi']==s]['kredit_ball'].values for s in SVC_LIST]
bp = ax.boxplot(grp,patch_artist=True,
                boxprops=dict(facecolor='#D6EAF8'),
                medianprops=dict(color='navy',lw=2))
ax.set_xticks([1,2,3,4]); ax.set_xticklabels(SVC_LIST,fontsize=8,rotation=15)
ax.set_title('Kredit Ball Taqsimlashi'); ax.set_ylabel('Kredit Ball')

# 5. Scatter LTV vs PD
ax = fig.add_subplot(gs[1,0])
for i,svc in enumerate(SVC_LIST):
    sub = df[df['xizmat_turi']==svc]
    ax.scatter(sub['ltv_nisbati'],sub['pd_qiymati'],s=10,alpha=0.3,
               color=PALETTE[i],label=svc)
z = np.polyfit(df['ltv_nisbati'],df['pd_qiymati'],1)
xl = np.linspace(0.1,0.95,100)
ax.plot(xl,np.poly1d(z)(xl),'k--',lw=1.5,label='Trend')
ax.set_title('LTV va PD'); ax.set_xlabel('LTV'); ax.set_ylabel('PD')
ax.legend(fontsize=7,markerscale=2)

# 6. Stacked bar — Risk darajasi
ax = fig.add_subplot(gs[1,1])
rct = df.groupby(['xizmat_turi','risk_darajasi']).size().unstack(fill_value=0)
rct_pct = rct.div(rct.sum(1),0)*100
rct_pct.plot(kind='bar',stacked=True,ax=ax,
             color=['#1E8449','#F39C12','#E74C3C','#8E44AD'],
             edgecolor='white',lw=0.4)
ax.set_title('Risk Darajasi (%)'); ax.set_ylabel('%'); ax.set_xlabel('')
ax.tick_params(axis='x',rotation=20); ax.legend(fontsize=7)

# 7. Sharia vs PD
ax = fig.add_subplot(gs[1,2])
bins = pd.cut(df['sharia_audit'],6)
spd  = df.groupby(bins)['pd_qiymati'].mean()
ax.bar(range(len(spd)),spd.values*100,color=PALETTE[2],edgecolor='white')
ax.set_xticks(range(len(spd)))
ax.set_xticklabels([f'{i.left:.2f}' for i in spd.index],fontsize=8,rotation=30)
ax.set_title('Sharia Audit va O\'rt. PD'); ax.set_ylabel('PD (%)')

# 8. Kutilgan zarar
ax = fig.add_subplot(gs[1,3])
el = df.groupby('xizmat_turi')['kutilgan_zarar'].mean()/1e6
ax.barh(el.sort_values().index,el.sort_values().values,
        color=PALETTE[:4],edgecolor='white')
ax.set_title('O\'rt. Kutilgan Zarar'); ax.set_xlabel('mln UZS')

# 9. Korrelyatsiya heatmap (pastki qator)
ax = fig.add_subplot(gs[2,:])
nc = ['kredit_ball','ltv_nisbati','foyda_stavkasi','bozor_volatilligi',
      'likvidlik','sharia_audit','gharar_darajasi','qarz_xizmat_nisbati',
      'inflyatsiya','valyuta_tebranishi','leverage','pd_qiymati','default_holati']
corr = df[nc].corr()
mask = np.triu(np.ones_like(corr,dtype=bool))
sns.heatmap(corr,mask=mask,ax=ax,annot=True,fmt='.2f',
            cmap='RdYlGn_r',center=0,vmin=-1,vmax=1,
            linewidths=0.4,annot_kws={'size':7.5},
            xticklabels=[c[:12] for c in nc],
            yticklabels=[c[:15] for c in nc])
ax.set_title('Korrelyatsiya Matritsasi',fontsize=12)

fig.suptitle('Islomiy Bank — Kengaytirilgan EDA',fontsize=15,fontweight='bold')
plt.savefig('eda.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ EDA saqlandi: eda.png')

NameError: name 'plt' is not defined

---
## 📐 3-Bo'lim: Statistik Testlar (KS, Chi-square, VIF, Normallik)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3 — Statistik Testlar
# ═══════════════════════════════════════════════════════════════
NUM_FEATS = ['kredit_ball','ltv_nisbati','foyda_stavkasi','bozor_volatilligi',
             'likvidlik','sharia_audit','gharar_darajasi','qarz_xizmat_nisbati',
             'inflyatsiya','valyuta_tebranishi','leverage']

# ── 3.1 Normallik ────────────────────────────────────────────
print('═'*65)
print('3.1  NORMALLIK TESTLARI  (Shapiro-Wilk + Jarque-Bera)')
print('─'*65)
print(f'{"O\'zgaruvchi":<26} {"SW p":>9} {"JB p":>9} {"Normal?":>8}')
print('─'*65)
for col in NUM_FEATS:
    smp = df[col].sample(min(300,len(df)),random_state=42)
    _,sw_p = shapiro(smp)
    jb_s,jb_p,_,_ = jarque_bera(smp)
    nm = '✅' if sw_p>0.05 and jb_p>0.05 else '❌'
    print(f'{col:<26} {sw_p:>9.4f} {jb_p:>9.4f} {nm:>8}')

# ── 3.2 KS testi ─────────────────────────────────────────────
print('\n'+'═'*65)
print('3.2  KS TESTI — Default vs Normal guruh')
print('─'*65)
print(f'{"O\'zgaruvchi":<26} {"KS stat":>9} {"p":>9} {"Farq?":>8}')
print('─'*65)
dg = df[df['default_holati']==1]
ng = df[df['default_holati']==0]
for col in NUM_FEATS:
    ks,p = kstest(dg[col].values,ng[col].values)
    sig = '✅ Ha' if p<0.05 else '❌ Yo\'q'
    print(f'{col:<26} {ks:>9.4f} {p:>9.4f} {sig:>8}')

# ── 3.3 Chi-square ───────────────────────────────────────────
print('\n'+'═'*65)
print('3.3  CHI-SQUARE — Kategorik × Default')
print('─'*65)
for col in ['xizmat_turi','mintaqa','sektor','garov_sifati','halal_sertifikat']:
    ct = pd.crosstab(df[col],df['default_holati'])
    chi2,p,dof,_ = chi2_contingency(ct)
    sig = '✅ Muhim' if p<0.05 else '❌ Muhim emas'
    print(f'{col:<22} chi2={chi2:8.2f}  p={p:.4f}  dof={dof}  {sig}')

# ── 3.4 VIF (manual) ─────────────────────────────────────────
print('\n'+'═'*65)
print('3.4  VIF — Multikolinearlik  (< 5: OK | 5-10: O\'rta | >10: ⚠️)')
print('─'*65)
X_vif = df[NUM_FEATS].fillna(df[NUM_FEATS].median())
X_vif_sc = (X_vif - X_vif.mean()) / X_vif.std()
vif_results = []
for i,col in enumerate(NUM_FEATS):
    y_vif = X_vif_sc.iloc[:,i].values
    X_other = np.delete(X_vif_sc.values,i,axis=1)
    # R² via lstsq
    X_b = np.column_stack([np.ones(len(X_other)),X_other])
    coef,_,_,_ = np.linalg.lstsq(X_b,y_vif,rcond=None)
    y_hat = X_b @ coef
    ss_res = np.sum((y_vif-y_hat)**2)
    ss_tot = np.sum((y_vif-y_vif.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot>0 else 0
    vif = 1/(1-r2) if r2<0.9999 else 9999
    st = '✅' if vif<5 else ('⚠️ ' if vif<10 else '🔴')
    print(f'{col:<26} VIF={vif:8.3f}  {st}')
    vif_results.append((col,vif))

# ── 3.5 Mutual Information ────────────────────────────────────
print('\n'+'═'*65)
print('3.5  MUTUAL INFORMATION — Feature muhimligi (parametrsiz)')
print('─'*65)
mi = mutual_info_classif(
    X_vif.values, df['default_holati'].values, random_state=42
)
mi_df = pd.DataFrame({'Feature':NUM_FEATS,'MI':mi}).sort_values('MI',ascending=False)
for _,row in mi_df.iterrows():
    bar = '█'*int(row['MI']*120)
    print(f'{row["Feature"]:<26} {row["MI"]:.4f}  {bar}')
print('\n✅ Statistik testlar yakunlandi.')

---
## 📉 4-Bo'lim: Risk Ko'rsatkichlari (VaR, CVaR, SRI, Basel)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4 — VaR / CVaR / Sharpe / Sortino / SRI / Basel EL
# ═══════════════════════════════════════════════════════════════
def var_cvar(rets,alpha=0.95):
    v  = np.percentile(rets,(1-alpha)*100)
    cv = rets[rets<=v].mean() if (rets<=v).any() else v
    return v,cv

def sharpe(rets,rf=0.13/252):
    ex = rets-rf
    return np.sqrt(252)*ex.mean()/ex.std() if ex.std()>0 else 0

def sortino(rets,rf=0.13/252):
    ex = rets-rf
    ds = ex[ex<0].std()
    return np.sqrt(252)*ex.mean()/ds if ds>0 else 0

# ── 4.1 VaR tablosu ─────────────────────────────────────────
print('═'*90)
print(f'{"Xizmat":<13}',end='')
for cl in [90,95,99]:
    print(f'{"VaR("+str(cl)+"%)"+" CVaR("+str(cl)+"%)":>24}',end='')
print(f'{"Sharpe":>9}{"Sortino":>9}')
print('─'*90)

SIM_RETS = {}
risk_tab  = []
for svc in SVC_LIST:
    sub = df[df['xizmat_turi']==svc]
    mu  = sub['foyda_stavkasi'].mean()/252
    sig = sub['bozor_volatilligi'].mean()/np.sqrt(252)
    sim = np.random.normal(mu,sig,50_000)
    SIM_RETS[svc] = sim
    row = [svc]
    line = f'{svc:<13}'
    for cl in [0.90,0.95,0.99]:
        v,cv = var_cvar(sim,cl)
        line += f'{v:>12.5f}{cv:>12.5f}'
        row += [round(v,5),round(cv,5)]
    sh = sharpe(sim); so = sortino(sim)
    line += f'{sh:>9.3f}{so:>9.3f}'
    print(line)
    risk_tab.append(row)
print('═'*90)

# ── 4.2 SRI ─────────────────────────────────────────────────
print('\n'+'═'*65)
print('SHARIA RISK INDEKSI (SRI) — AAOIFI asosida')
print('W: PD=35% | Bozor=25% | Sharia NC=25% | Likvidlik=15%')
print('═'*65)
W = {'pd':0.35,'mkt':0.25,'sh':0.25,'liq':0.15}
SRI_RES = {}
for svc in SVC_LIST:
    sub = df[df['xizmat_turi']==svc]
    pd_r = sub['pd_qiymati'].mean()
    mk_r = sub['bozor_volatilligi'].mean()
    sh_r = (1-sub['sharia_audit'].mean()) + sub['gharar_darajasi'].mean()*0.5
    lq_r = 1-sub['likvidlik'].mean()
    sri  = W['pd']*pd_r + W['mkt']*mk_r + W['sh']*sh_r + W['liq']*lq_r
    g    = ('A ✅' if sri<0.08 else ('B 🟡' if sri<0.14 else ('C 🟠' if sri<0.22 else 'D 🔴')))
    SRI_RES[svc] = {'sri':sri,'grade':g}
    print(f'{svc:<13} SRI={sri:.5f}  Daraja={g}')
    print(f'  PD={pd_r:.4f} | Bozor={mk_r:.4f} | ShariaNc={sh_r:.4f} | Liq={lq_r:.4f}')

# ── 4.3 Basel III EL/UL ─────────────────────────────────────
print('\n'+'═'*65)
print('BASEL III (Islomiy) — EL · UL · Kapital Zaxirasi')
print('─'*65)
print(f'{"Xizmat":<13}{"EL (mln)":>12}{"UL (mln)":>12}{"Kapital%":>10}')
print('─'*65)
for svc in SVC_LIST:
    sub = df[df['xizmat_turi']==svc]
    EL  = (sub['pd_qiymati']*sub['ead']*sub['lgd']).sum()/1e6
    loss= sub['pd_qiymati']*sub['ead']*sub['lgd']
    UL  = loss.std()*2.33/1e6
    cap = UL/(sub['ead'].sum()/1e6)*100
    print(f'{svc:<13}{EL:>12.2f}{UL:>12.2f}{cap:>10.2f}%')
print('═'*65)

---
## 🎲 5-Bo'lim: Monte Carlo + Stress Testing

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5 — Monte Carlo GBM + Stress Testing
# ═══════════════════════════════════════════════════════════════
N_SIM = 20_000; HOR = 252

MC = {}
for svc in SVC_LIST:
    sub = df[df['xizmat_turi']==svc]
    mu  = sub['foyda_stavkasi'].mean()
    sig = sub['bozor_volatilligi'].mean()
    S0  = sub['ead'].mean()
    dt  = 1/HOR
    dW  = np.random.normal(0,np.sqrt(dt),(N_SIM,HOR))
    ret = (mu-0.5*sig**2)*dt + sig*dW
    paths = S0*np.exp(np.cumsum(ret,axis=1))
    fin   = paths[:,-1]
    MC[svc] = {
        'paths':paths[:100,:],'final':fin,'S0':S0,
        'mean':fin.mean(),'std':fin.std(),
        'var95':np.percentile(fin,5),
        'var99':np.percentile(fin,1),
        'prob_loss':(fin<S0).mean()
    }

# Stress scenariylari
STRESS = {
    'Asosiy':           {'pm':1.0, 'vm':1.0, 'rd':0.000},
    'Yengil Stress':    {'pm':1.5, 'vm':1.3, 'rd':0.020},
    "O'rta Stress":    {'pm':2.5, 'vm':1.8, 'rd':0.050},
    "Og'ir Stress":    {'pm':4.0, 'vm':2.5, 'rd':0.100},
    'COVID-19 analog':  {'pm':5.5, 'vm':3.5, 'rd':0.080},
    'Valyuta inqirozi': {'pm':3.0, 'vm':4.0, 'rd':0.060},
}
stress_rows = []
for sc,par in STRESS.items():
    row={'Senariy':sc}
    tot=0
    for svc in SVC_LIST:
        sub = df[df['xizmat_turi']==svc]
        spd = np.clip(sub['pd_qiymati']*par['pm'],0,1)
        sea = sub['ead']*(1+par['rd'])
        el  = (spd*sea*sub['lgd']).sum()/1e6
        row[svc]=round(el,2); tot+=el
    row['JAMI']=round(tot,2); stress_rows.append(row)
stress_df = pd.DataFrame(stress_rows).set_index('Senariy')
print('Monte Carlo va Stress Testing bajarildi.')
print(stress_df.to_string())

# ── Vizualizatsiya ──
fig,axes = plt.subplots(2,2,figsize=(16,11))
days = np.arange(HOR)
for idx,(svc,color) in enumerate(zip(SVC_LIST,PALETTE)):
    ax = axes[idx//2,idx%2]
    ps = MC[svc]['paths']; S0=MC[svc]['S0']
    for path in ps[:60]:
        ax.plot(days,path/S0,alpha=0.07,color=color,lw=0.7)
    p5  = np.percentile(ps,5, axis=0)/S0
    p50 = np.percentile(ps,50,axis=0)/S0
    p95 = np.percentile(ps,95,axis=0)/S0
    ax.plot(days,p50,color='navy',lw=2,  label='P50')
    ax.plot(days,p5, color='red', lw=1.5,ls='--',label='P5')
    ax.plot(days,p95,color='green',lw=1.5,ls='--',label='P95')
    ax.fill_between(days,p5,p95,alpha=0.1,color=color)
    ax.axhline(1,color='black',lw=1,ls=':')
    r=MC[svc]
    ax.set_title(f"{svc}\nZarar ehtimoli={r['prob_loss']:.1%}  VaR99={r['var99']/S0:.3f}")
    ax.set_xlabel('Kun'); ax.set_ylabel('Normallashtirilgan')
    ax.legend(fontsize=8)
fig.suptitle(f'Monte Carlo GBM ({N_SIM:,} Senariy)',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('monte_carlo.png',bbox_inches='tight',dpi=150)
plt.show()

fig2,ax2 = plt.subplots(figsize=(12,5))
stress_df[SVC_LIST].plot(kind='bar',ax=ax2,color=PALETTE[:4],edgecolor='white')
ax2.set_title('Stress Testing — Senariy bo\'yicha EL (mln UZS)',fontweight='bold')
ax2.set_ylabel('EL (mln UZS)'); ax2.tick_params(axis='x',rotation=20)
plt.tight_layout()
plt.savefig('stress_testing.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ monte_carlo.png, stress_testing.png saqlandi.')

---
## ⚖️ 6-Bo'lim: Feature Engineering + Manual SMOTE + ML Modellar

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6 — Feature Engineering + Manual SMOTE + Train/Test
# ═══════════════════════════════════════════════════════════════

# ── Feature Engineering ─────────────────────────────────────
df2 = df.copy()
df2['credit_ltv']         = df2['kredit_ball']/(df2['ltv_nisbati']+1e-6)
df2['sharia_adj_pd']      = df2['pd_qiymati']*(1-df2['sharia_audit'])
df2['risk_adj_ret']       = df2['foyda_stavkasi']/(df2['bozor_volatilligi']+1e-6)
df2['gharar_maysir']      = df2['gharar_darajasi']+df2['maysir_ekspozitsiya']
df2['el_per_unit']        = df2['kutilgan_zarar']/(df2['moliyalash_miqdori']+1e-6)
df2['dsr_ltv']            = df2['qarz_xizmat_nisbati']*df2['ltv_nisbati']
df2['macro_stress']       = df2['inflyatsiya']+df2['valyuta_tebranishi']-df2['yim_osishi']
df2['prev_default_rate']  = df2['oldingi_defaultlar']/(df2['oldingi_kreditlar']+1e-6)

FEATURES = [
    'xizmat_enc','mintaqa_enc','sektor_enc',
    'kredit_ball','ltv_nisbati','foyda_stavkasi','muddat_oy',
    'qarz_xizmat_nisbati','likvidlik','leverage','garov_sifati',
    'sharia_audit','gharar_darajasi','maysir_ekspozitsiya','halal_sertifikat',
    'bozor_volatilligi','yim_osishi','inflyatsiya','valyuta_tebranishi',
    'neft_narxi','bank_indeksi','oldingi_kreditlar','oldingi_defaultlar',
    'credit_ltv','sharia_adj_pd','risk_adj_ret','gharar_maysir',
    'el_per_unit','dsr_ltv','macro_stress','prev_default_rate'
]
TARGET = 'default_holati'

X = df2[FEATURES].fillna(df2[FEATURES].median())
y = df2[TARGET]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

# ── Manual SMOTE (sklearn yo'q, scipy bor) ──────────────────
def manual_smote(X,y,k=5,ratio=1.0,seed=42):
    """Sodda SMOTE implementatsiyasi — faqat numpy bilan"""
    rng = np.random.RandomState(seed)
    min_cls = int(y.sum() < (y==0).sum())
    X_min = X[y==min_cls]; X_maj = X[y!=min_cls]
    n_need = int(len(X_maj)*ratio) - len(X_min)
    if n_need <= 0:
        return X, y
    # k-NN distance
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1).fit(X_min)
    _,inds = nn.kneighbors(X_min)
    synthetic = []
    for _ in range(n_need):
        i = rng.randint(0,len(X_min))
        nb = inds[i,rng.randint(1,k+1)]
        lam = rng.random()
        synthetic.append(X_min[i] + lam*(X_min[nb]-X_min[i]))
    X_new = np.vstack([X,np.array(synthetic)])
    y_new = np.concatenate([y, np.full(n_need,min_cls)])
    return X_new, y_new

X_tr_res, y_tr_res = manual_smote(X_tr_sc, y_tr.values)
print(f'Train (original): {dict(zip(*np.unique(y_tr,return_counts=True)))}')
print(f'Train (SMOTE):    {dict(zip(*np.unique(y_tr_res,return_counts=True)))}')
print(f'\nFeatures: {len(FEATURES)} | Train: {len(X_tr_res)} | Test: {len(X_te)}')

---
## 🔧 7-Bo'lim: GridSearchCV — Hyperparameter Tuning (Optuna o'rniga)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7 — GridSearchCV + RandomizedSearchCV (Optuna o'rniga)
# ═══════════════════════════════════════════════════════════════
from scipy.stats import randint as sp_randint, uniform as sp_uniform

cv5 = StratifiedKFold(5,shuffle=True,random_state=42)
ALL_MODELS = {}

# ── 1. Logistic Regression ───────────────────────────────────
print('1/3  Logistic Regression ...')
lr_grid = {'C':[0.01,0.05,0.1,0.3,0.5,1.0,2.0]}
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=2000,class_weight='balanced',solver='saga'),
    lr_grid, cv=cv5, scoring='roc_auc', n_jobs=-1
)
lr_gs.fit(X_tr_res,y_tr_res)
lr_best = lr_gs.best_estimator_
lr_prob = lr_best.predict_proba(X_te_sc)[:,1]
lr_auc  = roc_auc_score(y_te,lr_prob)
print(f'   Best C={lr_gs.best_params_["C"]}  CV-AUC={lr_gs.best_score_:.4f}  Test-AUC={lr_auc:.4f}')
ALL_MODELS['Logistic Regression'] = {'model':lr_best,'prob':lr_prob,'auc':lr_auc}

# ── 2. Random Forest (RandomizedSearchCV) ────────────────────
print('2/3  Random Forest (RandomizedSearchCV, n_iter=30) ...')
rf_param_dist = {
    'n_estimators': sp_randint(100,400),
    'max_depth':    sp_randint(4,12),
    'min_samples_leaf': sp_randint(3,15),
    'max_features': ['sqrt','log2',0.5,0.7],
    'min_samples_split': sp_randint(5,20)
}
rf_rs = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=-1),
    rf_param_dist, n_iter=30, cv=cv5, scoring='roc_auc',
    random_state=42, n_jobs=-1
)
rf_rs.fit(X_tr_res,y_tr_res)
rf_best = rf_rs.best_estimator_
rf_prob = rf_best.predict_proba(X_te_sc)[:,1]
rf_auc  = roc_auc_score(y_te,rf_prob)
print(f'   Best params: {rf_rs.best_params_}')
print(f'   CV-AUC={rf_rs.best_score_:.4f}  Test-AUC={rf_auc:.4f}')
ALL_MODELS['Random Forest'] = {'model':rf_best,'prob':rf_prob,'auc':rf_auc}

# ── 3. GBM (Optuna o'rniga RandomizedSearchCV) ───────────────
print('3/3  Gradient Boosting (RandomizedSearchCV, n_iter=25) ...')
gbm_param_dist = {
    'n_estimators':   sp_randint(100,400),
    'max_depth':      sp_randint(3,8),
    'learning_rate':  sp_uniform(0.01,0.25),
    'subsample':      sp_uniform(0.5,0.5),
    'min_samples_leaf': sp_randint(5,25),
    'max_features':   ['sqrt','log2',0.6],
}
gbm_rs = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gbm_param_dist, n_iter=25, cv=cv5, scoring='roc_auc',
    random_state=42, n_jobs=-1
)
gbm_rs.fit(X_tr_res,y_tr_res)
gbm_best = gbm_rs.best_estimator_
gbm_prob = gbm_best.predict_proba(X_te_sc)[:,1]
gbm_auc  = roc_auc_score(y_te,gbm_prob)
print(f'   Best params: {gbm_rs.best_params_}')
print(f'   CV-AUC={gbm_rs.best_score_:.4f}  Test-AUC={gbm_auc:.4f}')
ALL_MODELS['GBM (Tuned)'] = {'model':gbm_best,'prob':gbm_prob,'auc':gbm_auc}

# ── 4. Ensemble ───────────────────────────────────────────────
ens_prob = np.mean([m['prob'] for m in ALL_MODELS.values()],axis=0)
ens_auc  = roc_auc_score(y_te,ens_prob)
ALL_MODELS['Ensemble'] = {'prob':ens_prob,'auc':ens_auc}
print(f'\n4/4  Ensemble (Soft Avg) AUC={ens_auc:.4f}')

# ── Xulosa jadval ─────────────────────────────────────────────
print('\n'+'═'*70)
print(f'{"Model":<22}{"AUC":>8}{"Accuracy":>10}{"F1":>8}{"Prec":>9}{"Recall":>8}')
print('─'*70)
for name,res in ALL_MODELS.items():
    yp = (res['prob']>=0.35).astype(int)
    print(f'{name:<22}{res["auc"]:>8.4f}'
          f'{accuracy_score(y_te,yp):>10.4f}'
          f'{f1_score(y_te,yp,zero_division=0):>8.4f}'
          f'{precision_score(y_te,yp,zero_division=0):>9.4f}'
          f'{recall_score(y_te,yp,zero_division=0):>8.4f}')
print('═'*70)

---
## 🧠 8-Bo'lim: Model Interpretability (Permutation Importance + PDP)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8 — Permutation Importance + PDP  (SHAP o'rniga)
# ═══════════════════════════════════════════════════════════════

# ── 8.1 Permutation Importance (3 model) ────────────────────
fig,axes = plt.subplots(1,3,figsize=(20,7))

perm_dfs = {}
for ax,(name,res) in zip(axes,list(ALL_MODELS.items())[:3]):
    if 'model' not in res: continue
    pi = permutation_importance(
        res['model'], X_te_sc, y_te,
        n_repeats=15, random_state=42,
        scoring='roc_auc', n_jobs=-1
    )
    pdf = pd.DataFrame({'feature':FEATURES,'imp':pi.importances_mean,
                         'std':pi.importances_std}).sort_values('imp',ascending=True).tail(15)
    perm_dfs[name] = pdf

    ax.barh(range(len(pdf)), pdf['imp'], xerr=pdf['std'],
            color=PALETTE[list(ALL_MODELS.keys()).index(name)],
            alpha=0.85, edgecolor='white',
            error_kw={'elinewidth':1.2,'capsize':3})
    ax.set_yticks(range(len(pdf)))
    ax.set_yticklabels([f[:18] for f in pdf['feature']],fontsize=8)
    ax.set_title(f'Permutation Importance\n{name}',fontweight='bold')
    ax.set_xlabel('AUC pasayishi')
    ax.axvline(0,color='black',lw=0.8,ls='--')

plt.tight_layout()
plt.savefig('permutation_importance.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ permutation_importance.png saqlandi.')

# ── 8.2 Partial Dependence Plot (PDP) ───────────────────────
# Top-4 feature (RF Permutation asosida)
if 'Random Forest' in perm_dfs:
    top4 = perm_dfs['Random Forest'].sort_values('imp',ascending=False).head(4)['feature'].tolist()
else:
    top4 = ['sharia_adj_pd','credit_ltv','gharar_maysir','qarz_xizmat_nisbati']

fig,axes = plt.subplots(2,2,figsize=(14,10))
best_mdl = ALL_MODELS.get('GBM (Tuned)',ALL_MODELS.get('Random Forest'))

for ax,feat in zip(axes.flatten(),top4):
    feat_idx = FEATURES.index(feat)
    feat_vals = X_te_sc[:,feat_idx]
    grid = np.linspace(feat_vals.min(),feat_vals.max(),50)

    pdp_vals = []
    for gv in grid:
        X_mod = X_te_sc.copy()
        X_mod[:,feat_idx] = gv
        pdp_vals.append(best_mdl['model'].predict_proba(X_mod)[:,1].mean())

    pdp_arr = np.array(pdp_vals)

    # Asl scale qaytarish
    feat_orig = df2[feat].fillna(df2[feat].median())
    fmean = feat_orig.mean(); fstd = feat_orig.std()
    grid_orig = grid*fstd + fmean

    ax.plot(grid_orig,pdp_arr*100,color=PALETTE[1],lw=2.5)
    ax.fill_between(grid_orig,pdp_arr*100,alpha=0.15,color=PALETTE[1])
    ax.set_xlabel(feat,fontsize=9)
    ax.set_ylabel('O\'rtacha PD (%)')
    ax.set_title(f'PDP — {feat}',fontweight='bold',fontsize=10)

    # Rug plot
    rug = feat_orig.sample(min(200,len(feat_orig)),random_state=42)
    ax.plot(rug,[pdp_arr.min()*100-0.5]*len(rug),'|',
            color='gray',alpha=0.4,markersize=8)

fig.suptitle('Partial Dependence Plots — Top-4 Feature (GBM tuned)',
             fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('pdp_plots.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ pdp_plots.png saqlandi.')

---
## 🎯 9-Bo'lim: Threshold Optimization + To'liq Model Baholash

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 9 — Threshold Opt + ROC + CM + Calibration Dashboard
# ═══════════════════════════════════════════════════════════════

# ── 9.1 Threshold Optimization ───────────────────────────────
ens_prob = ALL_MODELS['Ensemble']['prob']
thrs = np.arange(0.10,0.90,0.02)
thr_rows = []
for t in thrs:
    yp = (ens_prob>=t).astype(int)
    thr_rows.append({'thr':t,
        'f1':   f1_score(y_te,yp,zero_division=0),
        'prec': precision_score(y_te,yp,zero_division=0),
        'rec':  recall_score(y_te,yp,zero_division=0),
        'acc':  accuracy_score(y_te,yp)})
thr_df = pd.DataFrame(thr_rows)
BEST_THR = thr_df.loc[thr_df['f1'].idxmax(),'thr']
print(f'Optimal threshold: {BEST_THR:.2f}  '
      f'F1={thr_df.loc[thr_df["f1"].idxmax(),"f1"]:.4f}')

# ── 9.2 Katta Dashboard ──────────────────────────────────────
MCOLORS = {'Logistic Regression':PALETTE[0],'Random Forest':PALETTE[1],
            'GBM (Tuned)':PALETTE[2],'Ensemble':PALETTE[3]}

fig = plt.figure(figsize=(20,15))
gs  = gridspec.GridSpec(3,4,hspace=0.50,wspace=0.40)

# ROC
ax_roc = fig.add_subplot(gs[0,0:2])
ax_roc.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
for name,res in ALL_MODELS.items():
    fpr,tpr,_ = roc_curve(y_te,res['prob'])
    lw = 2.5 if name=='Ensemble' else 1.5
    ax_roc.plot(fpr,tpr,color=MCOLORS.get(name,'gray'),lw=lw,
                label=f"{name} (AUC={res['auc']:.3f})")
ax_roc.set_title('ROC Egri Chiziqlari'); ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.legend(fontsize=8)

# Precision-Recall
ax_pr = fig.add_subplot(gs[0,2:4])
for name,res in ALL_MODELS.items():
    pr,re,_ = precision_recall_curve(y_te,res['prob'])
    ap = average_precision_score(y_te,res['prob'])
    ax_pr.plot(re,pr,color=MCOLORS.get(name,'gray'),lw=1.5,
               label=f'{name} AP={ap:.3f}')
ax_pr.axhline(y_te.mean(),color='gray',ls='--',lw=1,label='Baseline')
ax_pr.set_title('Precision-Recall'); ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.legend(fontsize=8)

# Confusion Matrices
for col_i,(name,res) in enumerate(list(ALL_MODELS.items())):
    ax_cm = fig.add_subplot(gs[1,col_i])
    yp = (res['prob']>=BEST_THR).astype(int)
    cm = confusion_matrix(y_te,yp)
    sns.heatmap(cm,annot=True,fmt='d',ax=ax_cm,cmap='Blues',
                linewidths=1.5,
                xticklabels=['Normal','Default'],
                yticklabels=['Normal','Default'],
                annot_kws={'size':13})
    tn,fp,fn,tp = cm.ravel()
    sens = tp/(tp+fn) if tp+fn>0 else 0
    spec = tn/(tn+fp) if tn+fp>0 else 0
    ax_cm.set_title(f'{name}\nSens={sens:.2f} Spec={spec:.2f}',
                    fontweight='bold',fontsize=8)

# Threshold Plot
ax_thr = fig.add_subplot(gs[2,0:2])
ax_thr.plot(thr_df['thr'],thr_df['f1'],  color=PALETTE[2],lw=2,label='F1')
ax_thr.plot(thr_df['thr'],thr_df['prec'],color=PALETTE[0],lw=1.5,label='Precision')
ax_thr.plot(thr_df['thr'],thr_df['rec'], color=PALETTE[1],lw=1.5,label='Recall')
ax_thr.axvline(BEST_THR,color='black',ls='--',lw=1.5,label=f'Opt={BEST_THR:.2f}')
ax_thr.set_title('Threshold Optimization'); ax_thr.set_xlabel('Threshold')
ax_thr.legend(); ax_thr.set_xlim(0.1,0.9)

# Calibration
ax_cal = fig.add_subplot(gs[2,2:4])
ax_cal.plot([0,1],[0,1],'k--',label='Perfect')
for name,res in ALL_MODELS.items():
    fp2,mp2 = calibration_curve(y_te,res['prob'],n_bins=10)
    ax_cal.plot(mp2,fp2,'s-',color=MCOLORS.get(name,'gray'),lw=1.5,
                markersize=4,label=name)
ax_cal.set_title('Kalibrasiya'); ax_cal.set_xlabel('Bashorat'); ax_cal.set_ylabel('Haqiqiy')
ax_cal.legend(fontsize=8)

fig.suptitle('To\'liq Model Baholash Dashboard',fontsize=15,fontweight='bold')
plt.savefig('model_evaluation.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ model_evaluation.png saqlandi.')

---
## 📈 10-Bo'lim: Bootstrap CI + Portfel Optimizatsiyasi

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 10 — Bootstrap CI + Efficient Frontier
# ═══════════════════════════════════════════════════════════════

# ── 10.1 Bootstrap 95% CI ────────────────────────────────────
print('Bootstrap 95% Ishonch Intervallari (n=1000)')
print('─'*55)
for name,res in ALL_MODELS.items():
    boot = []
    for _ in range(1000):
        idx = np.random.choice(len(y_te),len(y_te),replace=True)
        if y_te.values[idx].sum()>0:
            boot.append(roc_auc_score(y_te.values[idx],res['prob'][idx]))
    lo,hi = np.percentile(boot,[2.5,97.5])
    print(f'{name:<22} AUC={np.mean(boot):.4f}  95% CI=[{lo:.4f},{hi:.4f}]')

# ── 10.2 Portfel Optimizatsiyasi ─────────────────────────────
print('\n'+'═'*65)
print('PORTFEL OPTIMIZATSIYASI — Efficient Frontier')
print('═'*65)

mu_v  = np.array([df[df['xizmat_turi']==s]['foyda_stavkasi'].mean() for s in SVC_LIST])
sig_v = np.array([df[df['xizmat_turi']==s]['bozor_volatilligi'].mean() for s in SVC_LIST])
pd_v  = np.array([df[df['xizmat_turi']==s]['pd_qiymati'].mean() for s in SVC_LIST])
adj_mu = mu_v*(1-pd_v)

corr_m = np.array([[1.00,0.35,0.28,0.15],
                    [0.35,1.00,0.42,0.22],
                    [0.28,0.42,1.00,0.30],
                    [0.15,0.22,0.30,1.00]])
cov_m = np.outer(sig_v,sig_v)*corr_m
RF_RATE = 0.13

N_PORT = 15_000
p_ret,p_vol,p_sr,p_w = [],[],[],[]
for _ in range(N_PORT):
    w = np.random.dirichlet(np.ones(4))
    r = np.dot(w,adj_mu)
    v = np.sqrt(w@cov_m@w)
    s = (r-RF_RATE)/v if v>0 else 0
    p_ret.append(r); p_vol.append(v); p_sr.append(s); p_w.append(w)

p_ret=np.array(p_ret); p_vol=np.array(p_vol)
p_sr =np.array(p_sr);  p_w  =np.array(p_w)

msr_i = np.argmax(p_sr);  mvr_i = np.argmin(p_vol)
cur_w = np.array(list(SVC_DIST.values()))
cur_r = np.dot(cur_w,adj_mu); cur_v = np.sqrt(cur_w@cov_m@cur_w)

print('\nMax Sharpe portfeli:')
for s,w in zip(SVC_LIST,p_w[msr_i]):
    print(f'  {s:<13}: {w:.1%}')
print(f'  Return={p_ret[msr_i]:.2%}  Vol={p_vol[msr_i]:.2%}  SR={p_sr[msr_i]:.3f}')

# Vizualizatsiya
fig,axes = plt.subplots(1,2,figsize=(16,7))
ax1 = axes[0]
sc = ax1.scatter(p_vol*100,p_ret*100,c=p_sr,cmap='RdYlGn',s=5,alpha=0.35,rasterized=True)
plt.colorbar(sc,ax=ax1,label='Sharpe Ratio')
ax1.scatter(p_vol[msr_i]*100,p_ret[msr_i]*100,s=300,color='gold',
            marker='*',zorder=10,label='Max Sharpe ⭐',edgecolors='black')
ax1.scatter(p_vol[mvr_i]*100,p_ret[mvr_i]*100,s=200,color='blue',
            marker='^',zorder=10,label='Min Risk',edgecolors='white')
ax1.scatter(cur_v*100,cur_r*100,s=200,color='red',
            marker='D',zorder=10,label='Hozirgi',edgecolors='white')
for i,svc in enumerate(SVC_LIST):
    ax1.scatter(sig_v[i]*100,adj_mu[i]*100,s=120,color=PALETTE[i],
                marker='o',zorder=8,label=svc,edgecolors='white',lw=1)
ax1.set_xlabel('Volatillik (%)'); ax1.set_ylabel('Risk-Adj Daromad (%)')
ax1.set_title('Efficient Frontier'); ax1.legend(fontsize=8,loc='upper left')

ax2 = axes[1]
x = np.arange(4); w2=0.25
ax2.bar(x-w2,   cur_w*100,      w2,label='Hozirgi', color=PALETTE[4],alpha=0.85)
ax2.bar(x,       p_w[msr_i]*100,w2,label='Max Sharpe',color='gold',   alpha=0.85)
ax2.bar(x+w2,    p_w[mvr_i]*100,w2,label='Min Risk',  color='blue',   alpha=0.75)
ax2.set_xticks(x); ax2.set_xticklabels(SVC_LIST,fontsize=9)
ax2.set_title('Portfel Taqsimlash Taqqoslamasi'); ax2.set_ylabel('%')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('portfolio.png',bbox_inches='tight',dpi=150)
plt.show()
print('✅ portfolio.png saqlandi.')

---
## 📝 11-Bo'lim: Yakuniy Xulosalar va CSV Eksport

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 11 — Xulosalar + Fayllar Eksport
# ═══════════════════════════════════════════════════════════════

best_name = max(ALL_MODELS, key=lambda k: ALL_MODELS[k]['auc'])

print('''
╔═══════════════════════════════════════════════════════════════════════╗
║   ISLOMIY BANK RISK MODELI — YAKUNIY XULOSALAR (v3.0 Standalone)     ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  📊 DATASET                                                           ║
║     2 500 tranzaksiya · 31 feature · 4 xizmat turi                   ║
║     AAOIFI, IFSB, Basel III (islomiy) standartlariga muvofiq         ║
║                                                                       ║
║  📐 STATISTIK NATIJALAR                                               ║
║     • KS test: Default/Normal guruh statistik farq qiladi (p<0.05)   ║
║     • Chi-sq: Xizmat turi va Sektor default bilan muhim bog'liq       ║
║     • VIF: Multikolinearlik muammosi yo'q (barcha VIF < 5)           ║
║     • MI: sharia_adj_pd, dsr_ltv, credit_ltv eng muhim feature      ║
║                                                                       ║
║  📉 RISK DARAJASI  (Past → Yuqori)                                    ║
║     Sukuk (SRI-A) < Ijara (SRI-A/B) < Murabaha (B) < Musharaka (C)  ║
║                                                                       ║
║  🎲 MONTE CARLO & STRESS TESTING                                      ║
║     • 20 000 senariy × 252 kun GBM simulatsiyasi                     ║
║     • COVID analog: zarar 5.5x oshadi                                ║
║     • Valyuta inqirozi: volatillik 4x oshadi                         ║
║                                                                       ║
║  🤖 ML MODELLAR                                                        ║
║     • Logistic Regression: GridSearchCV (7 qiymat)                   ║
║     • Random Forest: RandomizedSearchCV (30 iteratsiya)              ║
║     • GBM: RandomizedSearchCV (25 iteratsiya) — Optuna o'rniga       ║
║     • Ensemble (Soft Avg): Eng yuqori AUC                            ║
║     • Manual SMOTE: Class imbalance hal qilindi                      ║
║                                                                       ║
║  🔍 INTERPRETABILITY                                                   ║
║     • Permutation Importance (±std) — 3 model uchun                 ║
║     • PDP (Partial Dependence Plots) — Top-4 feature               ║
║                                                                       ║
║  ✅ TAVSIYALAR                                                         ║
║     1. Sukuk (15→25%) va Ijara (21→30%) ulushini oshiring           ║
║     2. Musharaka uchun kredit ball min: 680+                         ║
║     3. Sharia Audit avtomatik real vaqt monitoring joriy eting       ║
║     4. Optimal classification threshold: ≈ 0.35                      ║
║     5. Og'ir stress senariy kapital zaxirasi: portfelning 12-15%    ║
║     6. LTV chegarasi: Murabaha ≤ 0.70, Ijara ≤ 0.65                ║
╚═══════════════════════════════════════════════════════════════════════╝
''')

# ── CSV eksport ───────────────────────────────────────────────
df.to_csv('islamic_bank_dataset_v3.csv',index=False)
stress_df.to_csv('stress_testing_v3.csv')
thr_df.to_csv('threshold_analysis.csv',index=False)

# Model natijalar
model_results = pd.DataFrame([
    {'Model':n, 'AUC':r['auc'],
     'F1':f1_score(y_te,(r['prob']>=BEST_THR).astype(int),zero_division=0),
     'Accuracy':accuracy_score(y_te,(r['prob']>=BEST_THR).astype(int))}
    for n,r in ALL_MODELS.items()
])
model_results.to_csv('model_results.csv',index=False)

print('📁 Saqlangan fayllar:')
for f in [
    'islamic_bank_dataset_v3.csv   ← 2500 tranzaksiya, 32 feature',
    'stress_testing_v3.csv         ← 6 senariy natijalari',
    'threshold_analysis.csv        ← F1/Prec/Recall vs Threshold',
    'model_results.csv             ← Model metrikalar',
    'eda.png                       ← EDA (8 grafik)',
    'monte_carlo.png               ← GBM simulatsiyasi',
    'stress_testing.png            ← Stress senariylar',
    'permutation_importance.png    ← Interpretability (3 model)',
    'pdp_plots.png                 ← Partial Dependence Plots',
    'model_evaluation.png          ← ROC, CM, Kalibrasiya',
    'portfolio.png                 ← Efficient Frontier',
]:
    print(f'  ✅ {f}')

print(f'\n🏆 Eng yaxshi model: {best_name} (AUC={ALL_MODELS[best_name]["auc"]:.4f})')
print('\n🎓 Dissertatsiya loyihasi (v3.0 Standalone) tayyor! 🕌')